# MOF-74-Mn — Bulk Periodic Structure
## Energy, Forces, Geometry Optimisation and Electronic Structure

---

### What is this notebook about?

This is the **third and most complex level** of the MOF-74-Mn series:

| Notebook | System | Methods |
|---|---|---|
| 1 · `dobdc_ligand_properties.ipynb` | Isolated H₄DOBDC | RDKit, xTB, ORCA (molecule) |
| 2 · `mn_dobdc_complex.ipynb` | Mn-DOBDC cluster | xTB UHF, ORCA mult=6 |
| **3 · This notebook** | **Bulk periodic MOF-74-Mn** | **MACE + CP2K periodic DFT** |

---

### What makes a periodic material different?

In an **isolated molecule** (notebooks 1 & 2), the structure is finite: it has edges, and
properties are described by discrete orbitals (HOMO, LUMO).

In a **crystal** like MOF-74-Mn:
- The structure repeats infinitely in 3-D via **periodic boundary conditions (PBC)**
- The minimum non-repeating unit is the **unit cell**, described by 6 lattice parameters (a, b, c, α, β, γ)
- Molecular orbitals become **bands** — continuous manifolds of states labelled by crystal momentum **k**
- The energy gap between bands is the **band gap** (not HOMO-LUMO gap)
- Properties depend on both the cell geometry and the PBC-wrapped neighbour interactions

---

### MOF-74 (CPO-27) Crystal Structure

```
Space group  :  R-3  (rhombohedral / trigonal, #148)
Topology     :  Hexagonal rod-packing (hxl net)
Metal SBU    :  1-D Mn₂-O₂ chains running along the c-axis
Pore shape   :  1-D hexagonal channels (~11 Å pore diameter)
Pore access  :  One open metal site per Mn (axial water desorbs on activation)

Conventional hexagonal cell:
  a = b ≈ 26.1 Å   c ≈ 6.8 Å   α = β = 90°  γ = 120°
  ~102 atoms per conventional cell; ~34 in the primitive rhombohedral cell
```

---

### Where to get the crystal structure

This notebook loads the actual experimental/DFT-validated structure — **we never
build it from scratch**. The geometry matters: the pore diameter, Mn coordination,
and rod connectivity are all fixed by the real crystal structure and cannot be
reliably approximated from a single molecule.

A real CIF is bundled with this repo at
[`notebooks/data/CPO27_Mn_qmof-75309dd.cif`](data/CPO27_Mn_qmof-75309dd.cif):
the DFT-relaxed (PBE-D3BJ) structure of dehydrated CPO-27-Mn / Mn-MOF-74 from the
**QMOF Database** (`qmof_id = qmof-75309dd`, Rosen et al., *Matter* 2021 /
*npj Comput. Mater.* 2022), itself derived from the original experimental
structure deposited in the CSD as **COKNUN01** (Valenzano et al.,
*J. Mater. Chem. A*, [10.1039/C4TA05794E](https://doi.org/10.1039/C4TA05794E)).
It is a 54-atom primitive cell (Mn₆C₂₄H₆O₁₈), space group *R*-3 (#148), with
PLD ≈ 11.1 Å and LCD ≈ 11.8 Å — matching the literature pore diameter.

| Source | Access | What you get |
|---|---|---|
| **Bundled CIF (default)** | Already in this repo | Real DFT-relaxed CPO-27-Mn, verified — **used unless overridden below** |
| **QMOF (live)** | Free, `pip install qmof` | Same database, useful if you want a *different* MOF or fresh metadata |
| **Materials Project** | Free API key from mp.org | DFT-relaxed structure, band gap, formation energy |
| **CSD** | Licence required | Gold-standard experimental CIF, for cross-checking |

**Workflow:** the `fetch_structure` cell below loads `CIF_PATH` (the bundled CIF
by default). If you clear `CIF_PATH`, it falls back to a live QMOF search, then
Materials Project if you supply an API key. If none of those resolve, it raises
an error rather than guessing a structure.


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# ── Structure source (tried in order) ─────────────────────────────────────────
CIF_PATH    = "data/CPO27_Mn_qmof-75309dd.cif"
                     # Bundled real CIF (default): DFT-relaxed CPO-27-Mn / Mn-MOF-74
                     #   from the QMOF Database (qmof_id=qmof-75309dd), derived from
                     #   CSD entry COKNUN01. See the markdown cell above for provenance.
                     # Set to None (or another path) to skip this and try the sources below.

MP_API_KEY  = None   # Option B: Materials Project API key
                     #   get one free at https://materialsproject.org/api
MP_MOF_ID   = "mp-1143752"  # MP ID for Mn-MOF-74 (verify at mp.org)

USE_QMOF    = True   # Option C: live search of the QMOF database (pip install qmof)
                     #   free, no API key, ships DFT properties too — only used if
                     #   CIF_PATH is unset/missing

# ── Calculation settings ───────────────────────────────────────────────────────
CUTOFF      = 6.0    # neighbour-list cutoff (Å) — use ≥6 for large MOF cells
MACE_MODEL  = "small"
CP2K_CMD       = None   # "/path/to/cp2k_shell.psmp" — periodic DFT (Section 5)
CP2K_XC        = "PBE"  # exchange-correlation functional (GPW / GTH pseudopotentials)
CP2K_CUTOFF_RY = 400.0  # planewave auxiliary-grid cutoff (Ry)
WORK_DIR    = "mof74_work"
FMAX        = 0.10   # BFGS convergence (eV/Å)

# ── Visualisation (Section 2.2 · tiled crystal view) ────────────────────────────
SUPERCELL_REPS      = (2, 2, 2)   # unit cells shown along a, b, c
SUPERCELL_VIEW_MODE = "spheres"   # "spheres"       — atoms only, no bonds (clean, no clutter)
                                   # "ball_and_stick" — atoms + bonds (like the single-cell view)
SUPERCELL_SHOW_BOXES = True       # draw a wireframe box per repeated unit cell


In [ ]:
import os, re, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ase, ase.io
from ase import Atoms
from ase.optimize import BFGS
from ase.neighborlist import NeighborList
from ase.cell import Cell
from goal.ml.data.graph import AtomicGraph

os.makedirs(WORK_DIR, exist_ok=True)
warnings.filterwarnings("ignore", category=UserWarning)

try:
    import py3Dmol; HAS_PY3DMOL = True
except ImportError:
    HAS_PY3DMOL = False

import torch
# mace-torch pins e3nn==0.4.4, whose bundled Wigner-3j constants are loaded with a
# bare torch.load() call. Since torch>=2.6 defaults torch.load to weights_only=True,
# that call now raises UnpicklingError on import unless we allowlist `slice` first.
torch.serialization.add_safe_globals([slice])
try:
    from mace.calculators import mace_mp; HAS_MACE = True
except ImportError:
    HAS_MACE = False
    print("MACE missing — install into an isolated env, see the `mace-cuda` pixi "
          "environment / [mace] extra documented in pyproject.toml")

try:
    from ase.calculators.cp2k import CP2K; HAS_CP2K = True
except ImportError:
    HAS_CP2K = False

CPK = {
    "H":"#FFFFFF","C":"#808080","N":"#4169E1","O":"#FF4500","S":"#FFD700",
    "Mn":"#9C3E8F","Fe":"#E06000","Co":"#0000C8","Ni":"#28A028",
    "Cu":"#C88033","Mg":"#8AFF00","Ca":"#3DFF00","Zn":"#7D80B0",
}
VDW = {"H":0.31,"C":0.77,"N":0.75,"O":0.73,"Mn":1.61,"Fe":1.56,
       "Co":1.52,"Ni":1.49,"Cu":1.45,"Mg":1.60,"Zn":1.42}
print("Imports done.")


## Section 2 · Crystal Structure

### 2.0 · Load from CIF (bundled real structure → live QMOF → Materials Project)


In [ ]:
# ── Database loading helpers ──────────────────────────────────────────────────
import pathlib

def _show_structure(atoms, source):
    """Print a brief summary after loading."""
    sym = atoms.get_chemical_symbols()
    from collections import Counter
    comp = Counter(sym)
    l = atoms.cell.lengths(); a = atoms.cell.angles()
    print(f"✓ Loaded from: {source}")
    print(f"  Atoms        : {len(atoms)}")
    print(f"  Composition  : {dict(comp)}")
    print(f"  a,b,c (Å)    : {l[0]:.3f}  {l[1]:.3f}  {l[2]:.3f}")
    print(f"  α,β,γ  (°)   : {a[0]:.2f}  {a[1]:.2f}  {a[2]:.2f}")
    print(f"  Volume (Å³)  : {atoms.get_volume():.1f}")


# ════════════════════════════════════════════════════════════════════════════
# Option A — local CIF (bundled real structure by default, or a user path)
# ════════════════════════════════════════════════════════════════════════════
atoms_mof   = None
qmof_props  = {}   # will hold pre-computed DFT properties if QMOF is used

if CIF_PATH is not None:
    cif_p = pathlib.Path(CIF_PATH)
    if cif_p.exists():
        atoms_mof = ase.io.read(str(cif_p))
        _show_structure(atoms_mof, f"local CIF  {cif_p}")
    else:
        print(f"✗ CIF_PATH set but file not found: {cif_p}")


# ════════════════════════════════════════════════════════════════════════════
# Option B — QMOF Database, live search  (pip install qmof)
#   • Free, no API key
#   • ~20 000 MOFs with DFT-computed properties (band gap, DOS, charges)
#   • Reference: Rosen et al., Matter 2021, https://doi.org/10.1016/j.matt.2021.02.015
# ════════════════════════════════════════════════════════════════════════════
if atoms_mof is None and USE_QMOF:
    try:
        import qmof                                  # pip install qmof
        print("Searching QMOF database for MOF-74-Mn / CPO-27-Mn …")

        df = qmof.get_qmof_df()                      # downloads index (~few MB)

        mask = (
            df["name"].str.contains("CPO-27-Mn", case=False, na=False) |
            df["name"].str.contains("MOF-74-Mn", case=False, na=False) |
            df["name"].str.contains("COKNUN",    case=False, na=False)
        )
        hits = df[mask]
        print(f"  Matches found: {len(hits)}")

        if len(hits) > 0:
            row = hits.iloc[0]
            print(f"  Using: {row['name']}  (qmof_id={row.get('qmof_id', 'N/A')})")

            cif_path_qmof = qmof.get_cif(row["qmof_id"], download_dir=WORK_DIR)
            atoms_mof = ase.io.read(cif_path_qmof)

            for key in ["pbe_bandgap", "hle17_bandgap", "charge_density_path",
                        "pbe_energy_per_atom", "pbe_absolute_magnetization"]:
                if key in row and row[key] is not None:
                    qmof_props[key] = row[key]

            _show_structure(atoms_mof, f"QMOF  {row.get('qmof_id','?')}")
            if qmof_props:
                print("\n  Pre-computed QMOF DFT properties:")
                for k, v in qmof_props.items():
                    print(f"    {k}: {v}")

    except ImportError:
        print("qmof package not installed — pip install qmof")
    except Exception as e:
        print(f"QMOF search failed: {e}")


# ════════════════════════════════════════════════════════════════════════════
# Option C — Materials Project  (needs API key + mp-api package)
#   • pip install mp-api
#   • Free key at: https://materialsproject.org/api
#   • Reference: Jain et al., APL Materials 2013, doi:10.1063/1.4812323
# ════════════════════════════════════════════════════════════════════════════
if atoms_mof is None and MP_API_KEY is not None:
    try:
        from mp_api.client import MPRester
        from pymatgen.io.ase import AseAtomsAdaptor

        print(f"Querying Materials Project  ({MP_MOF_ID}) …")
        with MPRester(MP_API_KEY) as mpr:
            structure = mpr.get_structure_by_material_id(MP_MOF_ID)
        atoms_mof = AseAtomsAdaptor.get_atoms(structure)
        _show_structure(atoms_mof, f"Materials Project  {MP_MOF_ID}")
    except ImportError:
        print("mp-api / pymatgen not installed — pip install mp-api pymatgen")
    except Exception as e:
        print(f"Materials Project query failed: {e}")


# ════════════════════════════════════════════════════════════════════════════
# No structure resolved — do NOT guess one. Fail loudly instead.
# ════════════════════════════════════════════════════════════════════════════
if atoms_mof is None:
    raise RuntimeError(
        "No real crystal structure could be loaded. This notebook refuses to "
        "fall back to a hand-built/guessed geometry — the pore size, Mn "
        "coordination, and rod connectivity all depend on the real structure. "
        "Fix one of:\n"
        f"  • CIF_PATH points at a real file (default: {CIF_PATH})\n"
        "  • USE_QMOF=True and `pip install qmof` for a live search\n"
        "  • MP_API_KEY set to a valid Materials Project key"
    )

os.makedirs(WORK_DIR, exist_ok=True)
atoms_mof.write(f"{WORK_DIR}/mof74_mn_structure.xyz")
print(f"\nStructure saved → {WORK_DIR}/mof74_mn_structure.xyz")


In [ ]:
sym  = atoms_mof.get_chemical_symbols()
pos  = atoms_mof.get_positions()
cell = atoms_mof.get_cell()

from collections import Counter
comp = Counter(sym)
print("── Structure Summary ───────────────────────────────────────")
print(f"  Total atoms  : {len(atoms_mof)}")
print(f"  Composition  : {dict(comp)}")
print(f"  PBC          : {atoms_mof.get_pbc().tolist()}")
print()
lengths = cell.lengths(); angles = cell.angles()
print(f"  a = {lengths[0]:.3f} Å    α = {angles[0]:.2f}°")
print(f"  b = {lengths[1]:.3f} Å    β = {angles[1]:.2f}°")
print(f"  c = {lengths[2]:.3f} Å    γ = {angles[2]:.2f}°")
print(f"  Volume  : {cell.volume:.2f} Å³")
print()
n_mn = comp.get("Mn", 0)
print(f"  Mn atoms per cell : {n_mn}")
if n_mn > 0:
    # Mn–Mn distances
    mn_pos_list = [pos[i] for i,s in enumerate(sym) if s=="Mn"]
    if len(mn_pos_list) > 1:
        mn_arr = np.array(mn_pos_list)
        for i in range(len(mn_arr)):
            for j in range(i+1, len(mn_arr)):
                d = np.linalg.norm(mn_arr[i]-mn_arr[j])
                if d < 5.0:
                    print(f"  Mn–Mn: {d:.3f} Å")


### 2.1 · 3D Visualisation with Unit Cell Box


In [ ]:
if not HAS_PY3DMOL:
    print("py3Dmol not available"); raise SystemExit

_syms = atoms_mof.get_chemical_symbols()
_pos  = atoms_mof.get_positions()
_cell = atoms_mof.get_cell()

# Bond detection (PBC-aware NeighborList)
_rc = [VDW.get(s, 0.8) for s in _syms]
_nl = NeighborList(_rc, skin=0.25, self_interaction=False, bothways=False)
_nl.update(atoms_mof)

view = py3Dmol.view(width=750, height=520)
view.setBackgroundColor("#0d0d1a")

# Atoms
for i, (sym, p) in enumerate(zip(_syms, _pos)):
    col = CPK.get(sym, "#AAAAAA")
    rad = VDW.get(sym, 0.8) * (0.50 if sym in ("Mn","Fe","Co") else 0.28)
    view.addSphere({"center":{"x":float(p[0]),"y":float(p[1]),"z":float(p[2])},
                    "radius":rad, "color":col, "opacity":1.0})

# Bonds (molecular bonds, no PBC wrapping in py3Dmol)
for i in range(len(_syms)):
    idxs, _ = _nl.get_neighbors(i)
    for j in idxs:
        p1, p2 = _pos[i], _pos[j]
        mid3 = (p1+p2)/2
        c1, c2 = CPK.get(_syms[i],"#888"), CPK.get(_syms[j],"#888")
        for (st, en, col) in [(p1, mid3, c1), (mid3, p2, c2)]:
            view.addCylinder({"start":{"x":float(st[0]),"y":float(st[1]),"z":float(st[2])},
                              "end":  {"x":float(en[0]),"y":float(en[1]),"z":float(en[2])},
                              "radius":0.06, "color":col, "opacity":0.9})

# ── Unit cell box ─────────────────────────────────────────────────────────────
# Corners: index 0..7 where bit 2 = along a, bit 1 = along b, bit 0 = along c
_a, _b, _c = np.array(_cell[0]), np.array(_cell[1]), np.array(_cell[2])
_orig = np.zeros(3)

def _corner(ia, ib, ic):
    return _orig + ia*_a + ib*_b + ic*_c

_corners = [_corner((i>>2)&1, (i>>1)&1, i&1) for i in range(8)]
_edges = [(0,1),(0,2),(0,4),(1,3),(1,5),(2,3),(2,6),(3,7),(4,5),(4,6),(5,7),(6,7)]

for ei, ej in _edges:
    p1, p2 = _corners[ei], _corners[ej]
    view.addCylinder({
        "start":{"x":float(p1[0]),"y":float(p1[1]),"z":float(p1[2])},
        "end":  {"x":float(p2[0]),"y":float(p2[1]),"z":float(p2[2])},
        "radius":0.04, "color":"#88AAEE", "opacity":0.7
    })

# Mn labels
for i, (sym, p) in enumerate(zip(_syms, _pos)):
    if sym == "Mn":
        view.addLabel("Mn", {"position":{"x":float(p[0]),"y":float(p[1]),"z":float(p[2])},
                              "fontColor":"white","backgroundColor":"transparent",
                              "fontSize":11})

view.zoomTo(); view.show()


### 2.2 · 3D Visualisation — Tiled Crystal (PBC repeated along a, b, c)

The single unit cell above is one box; the real crystal is that box repeated
infinitely. Here it's repeated `SUPERCELL_REPS` times along each lattice vector
so the periodicity is visible — the reference cell is highlighted in gold.

`SUPERCELL_VIEW_MODE` (set above in the config cell) controls the rendering:
- `"spheres"` — atoms only, no bond cylinders. With many repeated cells, bonds
  between every tiled copy get visually noisy fast; this mode just shows the
  node positions cleanly.
- `"ball_and_stick"` — atoms + bonds, same style as the single-cell view.


In [ ]:
if not HAS_PY3DMOL:
    print("py3Dmol not available"); raise SystemExit

_syms0 = atoms_mof.get_chemical_symbols()
_pos0  = atoms_mof.get_positions()
_cell  = atoms_mof.get_cell()
_a, _b, _c = np.array(_cell[0]), np.array(_cell[1]), np.array(_cell[2])

NX, NY, NZ = SUPERCELL_REPS

# ── Tile the unit cell along a, b, c to show the crystal repeating via PBC ────
_syms, _pos, _is_ref_cell = [], [], []
for i in range(NX):
    for j in range(NY):
        for k in range(NZ):
            shift = i * _a + j * _b + k * _c
            for s, p in zip(_syms0, _pos0):
                _syms.append(s)
                _pos.append(p + shift)
                _is_ref_cell.append(i == 0 and j == 0 and k == 0)
_pos = np.array(_pos)

atoms_super = Atoms(_syms, positions=_pos, cell=_cell, pbc=True)
print(f"Supercell: {NX}×{NY}×{NZ} unit cells → {len(atoms_super)} atoms "
      f"(from {len(atoms_mof)} in the unit cell)  ·  view mode: {SUPERCELL_VIEW_MODE!r}")

view = py3Dmol.view(width=850, height=600)
view.setBackgroundColor("#0d0d1a")

# Atoms
for sym, p in zip(_syms, _pos):
    col = CPK.get(sym, "#AAAAAA")
    rad = VDW.get(sym, 0.8) * (0.50 if sym in ("Mn","Fe","Co") else 0.28)
    view.addSphere({"center":{"x":float(p[0]),"y":float(p[1]),"z":float(p[2])},
                    "radius":rad, "color":col, "opacity":1.0})

# Bonds — only in "ball_and_stick" mode. Neighbouring tiled copies are now real
# atoms, so bonds crossing the original unit-cell boundary connect correctly.
if SUPERCELL_VIEW_MODE == "ball_and_stick":
    _rc = [VDW.get(s, 0.8) for s in _syms]
    _nl = NeighborList(_rc, skin=0.25, self_interaction=False, bothways=False)
    _nl.update(atoms_super)
    for i in range(len(_syms)):
        idxs, _ = _nl.get_neighbors(i)
        for j in idxs:
            p1, p2 = _pos[i], _pos[j]
            mid3 = (p1+p2)/2
            c1, c2 = CPK.get(_syms[i],"#888"), CPK.get(_syms[j],"#888")
            for (st, en, col) in [(p1, mid3, c1), (mid3, p2, c2)]:
                view.addCylinder({"start":{"x":float(st[0]),"y":float(st[1]),"z":float(st[2])},
                                  "end":  {"x":float(en[0]),"y":float(en[1]),"z":float(en[2])},
                                  "radius":0.06, "color":col, "opacity":0.9})
elif SUPERCELL_VIEW_MODE != "spheres":
    raise ValueError(f"Unknown SUPERCELL_VIEW_MODE: {SUPERCELL_VIEW_MODE!r} "
                      "(expected 'spheres' or 'ball_and_stick')")

# ── Unit cell boxes — one wireframe per repeated cell, so the tiling is visible ─
if SUPERCELL_SHOW_BOXES:
    def _corner(origin, ia, ib, ic):
        return origin + ia*_a + ib*_b + ic*_c

    _box_edges = [(0,1),(0,2),(0,4),(1,3),(1,5),(2,3),(2,6),(3,7),(4,5),(4,6),(5,7),(6,7)]

    for i in range(NX):
        for j in range(NY):
            for k in range(NZ):
                origin = i*_a + j*_b + k*_c
                corners = [_corner(origin, (c>>2)&1, (c>>1)&1, c&1) for c in range(8)]
                is_ref = (i == 0 and j == 0 and k == 0)
                col = "#FFD700" if is_ref else "#88AAEE"
                op  = 0.9 if is_ref else 0.4
                for ei, ej in _box_edges:
                    p1, p2 = corners[ei], corners[ej]
                    view.addCylinder({
                        "start":{"x":float(p1[0]),"y":float(p1[1]),"z":float(p1[2])},
                        "end":  {"x":float(p2[0]),"y":float(p2[1]),"z":float(p2[2])},
                        "radius":0.04, "color":col, "opacity":op
                    })

# Mn labels — only on the reference (highlighted) unit cell, to avoid clutter
for sym, p, is_ref in zip(_syms, _pos, _is_ref_cell):
    if sym == "Mn" and is_ref:
        view.addLabel("Mn", {"position":{"x":float(p[0]),"y":float(p[1]),"z":float(p[2])},
                              "fontColor":"white","backgroundColor":"transparent",
                              "fontSize":11})

view.zoomTo(); view.show()


## Section 3 · AtomicGraph (Periodic)

For periodic systems, `AtomicGraph.from_ase` automatically:
- Sets `pbc = True` along all periodic directions
- Adds **unit_shifts** to each edge (the integer vector of PBC image crossings)
- The GOAL models use `unit_shifts` to reconstruct the correct Cartesian displacement
  with the current cell vectors — essential for stress tensor calculations.


In [ ]:
graph = AtomicGraph.from_ase(atoms_mof, cutoff=CUTOFF)

print(f"AtomicGraph  (periodic={'✓' if graph.pbc.any() else '✗'})")
print(f"  Nodes (atoms)    : {graph.num_nodes}")
print(f"  Edges (pairs)    : {graph.num_edges}")
print(f"  edge_index shape : {graph.edge_index.shape}")
print(f"  unit_shifts shape: {graph.unit_shifts.shape}")
print(f"  cell shape       : {graph.cell.shape}")
print(f"  PBC              : {graph.pbc.tolist()}")

# How many edges cross a cell boundary?
n_cross = int((graph.unit_shifts.abs().sum(-1) > 0).sum())
print(f"\n  Cross-cell edges : {n_cross} / {graph.num_edges} "
      f"({100*n_cross/graph.num_edges:.1f}%)")
print("  → Non-zero unit_shifts confirm PBC is active")


In [ ]:
# Recompute edge vectors for analysis (graph stores unit_shifts, not vectors)
cell_mat = graph.cell.squeeze(0).numpy()            # (3,3) row vectors
pos_np   = graph.positions.numpy()                  # (N,3)
ei       = graph.edge_index.numpy()                 # (2,E)
us       = graph.unit_shifts.numpy()                # (E,3) integer

edge_vecs  = pos_np[ei[1]] - pos_np[ei[0]] + us @ cell_mat
edge_dists = np.linalg.norm(edge_vecs, axis=-1)

fig, ax = plt.subplots(figsize=(7, 3))
ax.set_facecolor("#1a1a2e"); fig.patch.set_facecolor("#1a1a2e")
for sp in ax.spines.values(): sp.set_color("#AAAAAA")
ax.tick_params(colors="#CCCCCC")
ax.set_title("Edge-length distribution (AtomicGraph, PBC active)", color="#CCCCCC")
ax.hist(edge_dists, bins=60, color="#9C3E8F", alpha=0.8, edgecolor="#333355")
ax.axvline(CUTOFF, color="red", lw=1.5, linestyle="--", label=f"cutoff {CUTOFF} Å")
ax.set_xlabel("Edge length (Å)", color="#CCCCCC")
ax.set_ylabel("Count", color="#CCCCCC")
ax.legend(facecolor="#2a2a4e", labelcolor="white")
plt.tight_layout(); plt.show()
print(f"Edge distances: min {edge_dists.min():.3f} Å  mean {edge_dists.mean():.3f} Å  max {edge_dists.max():.3f} Å")


## Section 4 · MACE-MP-0 Periodic Calculation

MACE-MP-0 was trained on the Materials Project database — a dataset of periodic
crystalline materials. It naturally supports PBC and can compute:
- **Energy**: total potential energy of the unit cell (eV)
- **Forces**: −∂E/∂r for each atom (eV/Å)
- **Stress tensor**: −(1/V) ∂E/∂ε (eV/Å³ = GPa / 160.2)

The stress tensor is important for:
- Geometry optimisation at constant pressure (NPT)
- Predicting mechanical properties (bulk modulus, elastic constants)


In [ ]:
if not HAS_MACE:
    raise RuntimeError("Install MACE: see the `mace-cuda` pixi environment in pyproject.toml")

device = "cuda" if torch.cuda.is_available() else "cpu"
mace_calc = mace_mp(model=MACE_MODEL, dispersion=False, device=device,
                    default_dtype="float32")
print(f"MACE-MP-0 ({MACE_MODEL}) loaded on {device}")


In [ ]:
atoms_mace = atoms_mof.copy()
atoms_mace.calc = mace_calc

e_mace  = atoms_mace.get_potential_energy()
f_mace  = atoms_mace.get_forces()
st_mace = atoms_mace.get_stress(voigt=False)   # (3,3) Cartesian stress

vol   = atoms_mace.get_volume()
GPa   = 160.2177                              # 1 eV/Å³ = 160.22 GPa
st_GPa = st_mace * GPa

print(f"MACE-MP-0  single-point (periodic)")
print(f"  Energy         : {e_mace:.4f} eV")
print(f"  Energy/atom    : {e_mace/len(atoms_mace):.4f} eV/atom")
print(f"  Volume         : {vol:.2f} Å³")
print(f"  |F|max         : {np.abs(f_mace).max():.4f} eV/Å")
print(f"  |F|rms         : {np.sqrt((f_mace**2).mean()):.4f} eV/Å")
print()
print("  Stress tensor (GPa):")
for row in st_GPa:
    print(f"    {row[0]:8.3f} {row[1]:8.3f} {row[2]:8.3f}")
print(f"  Hydrostatic pressure: {-np.trace(st_GPa)/3:.3f} GPa")


In [ ]:
# ── Geometry optimisation (NVT: fixed cell, relax atom positions) ─────────────
print("Running BFGS geometry optimisation with MACE (fixed cell) …")
atoms_opt = atoms_mof.copy()
atoms_opt.calc = mace_calc

# Use BFGS for NVT (constant volume) optimisation
opt = BFGS(atoms_opt, logfile=f"{WORK_DIR}/mof74_bfgs.log")
opt.run(fmax=FMAX, steps=400)

e_opt = atoms_opt.get_potential_energy()
f_opt = atoms_opt.get_forces()

print(f"\nAfter BFGS:")
print(f"  Energy        : {e_opt:.4f} eV  (was {e_mace:.4f} eV)")
print(f"  ΔE            : {e_opt - e_mace:.4f} eV")
print(f"  |F|max        : {np.abs(f_opt).max():.4f} eV/Å  (target {FMAX})")

atoms_opt.write(f"{WORK_DIR}/mof74_mn_mace_opt.xyz")
print(f"\nSaved → {WORK_DIR}/mof74_mn_mace_opt.xyz")
atoms_relax = atoms_opt.copy()   # use relaxed structure for subsequent analysis


## Section 5 · Periodic DFT via CP2K (GPW / PBE)

CP2K's Quickstep (GPW) module is a genuinely **periodic-native** DFT engine —
unlike ORCA, which is a molecular quantum-chemistry code with no PBC support at
all, CP2K solves the Kohn–Sham equations directly on the periodic cell using a
mixed Gaussian-basis / planewave (GPW) scheme with GTH pseudopotentials. That
makes it the natural swap-in for a periodic single-point check on this cell,
instead of the semi-empirical GFN2-xTB used previously.

Configure the binary via `CP2K_CMD` in the config cell above (e.g.
`"/path/to/cp2k_shell.psmp"`) — this section is skipped until it's set.

> **Note on Mn²⁺ spin state in the bulk:**
> As in the earlier semi-empirical estimate, the full unit cell contains
> multiple Mn²⁺ centres with 5 unpaired electrons each. For a quick UKS
> single-point we use the ferromagnetic-limit multiplicity `2S+1 = 5·n_Mn + 1`.
> This is an approximation; rigorous magnetic ordering requires broken-symmetry
> DFT with an explicit antiferromagnetic spin configuration.


In [ ]:
import ase.units as units

if not HAS_CP2K or CP2K_CMD is None:
    print("CP2K not configured — skipping periodic DFT "
          f"(set CP2K_CMD = '/path/to/cp2k_shell.psmp'; HAS_CP2K={HAS_CP2K})")
    raise SystemExit

n_mn = int((atoms_relax.get_atomic_numbers() == 25).sum())   # 25 = Mn
multiplicity_bulk = 5 * n_mn + 1                              # ferromagnetic estimate, 2S+1
charge_cell = 0

print(f"Periodic DFT (CP2K, GPW/{CP2K_XC}):")
print(f"  Atoms in cell  : {len(atoms_relax)}")
print(f"  Mn atoms       : {n_mn}")
print(f"  Multiplicity   : {multiplicity_bulk}  (ferromagnetic estimate, UKS)")
print(f"  Charge         : {charge_cell}")

cp2k_dir = f"{WORK_DIR}/cp2k_bulk"
os.makedirs(cp2k_dir, exist_ok=True)

calc_cp2k = CP2K(
    command=CP2K_CMD,
    xc=CP2K_XC,
    uks=True,
    charge=charge_cell,
    multiplicity=multiplicity_bulk,
    cutoff=CP2K_CUTOFF_RY * units.Rydberg,
    basis_set="DZVP-MOLOPT-SR-GTH",
    basis_set_file="BASIS_MOLOPT",
    pseudo_potential="auto",
    potential_file="POTENTIAL",
    stress_tensor=True,
    print_level="MEDIUM",
    label=f"{cp2k_dir}/mof74_bulk",
)

atoms_cp2k = atoms_relax.copy()
atoms_cp2k.calc = calc_cp2k

e_cp2k_bulk  = atoms_cp2k.get_potential_energy()
f_cp2k_bulk  = atoms_cp2k.get_forces()
st_cp2k_bulk = atoms_cp2k.get_stress(voigt=False) * 160.2177   # GPa

print(f"\nCP2K periodic energy : {e_cp2k_bulk:.4f} eV")
print(f"  per atom           : {e_cp2k_bulk/len(atoms_cp2k):.4f} eV/atom")
print(f"  |F|max             : {np.abs(f_cp2k_bulk).max():.4f} eV/Å")
print(f"  |F|rms             : {np.sqrt((f_cp2k_bulk**2).mean()):.4f} eV/Å")
print(f"  Hydrostatic stress : {-np.trace(st_cp2k_bulk)/3:.3f} GPa")


## Section 6 · Electronic Structure and Band Gap

### Important caveat

The CP2K run above is a **single-point energy/force/stress** calculation — it
does not, by itself, extract the crystal band gap:

| Quantity | Method | Meaning |
|---|---|---|
| Total energy (this notebook) | CP2K GPW/PBE single-point | Physically meaningful periodic DFT energy |
| Band gap (valence band max → conduction band min) | Periodic DFT with k-point sampling + `&PDOS`/`&PRINT &BAND_STRUCTURE` | Physically correct; not run automatically here |

For MOF-74-Mn, the experimental optical band gap is **~2.3 eV** (from diffuse
reflectance spectroscopy). Getting a comparable number out of CP2K needs an
explicit follow-up run with:

```
&DFT
  &PRINT
    &PDOS
      NLUMO -1
    &END PDOS
  &END PRINT
&END DFT
```

or a full `&BAND_STRUCTURE` section with an explicit k-point path — out of
scope for a single-point notebook cell, so it isn't fabricated here.


In [ ]:
# ── Compare CP2K periodic DFT energy with pre-computed QMOF DFT values (if available) ──
if qmof_props:
    print("── QMOF Pre-computed DFT Properties ─────────────────────────")
    pbe_gap  = qmof_props.get("pbe_bandgap")
    hle_gap  = qmof_props.get("hle17_bandgap")
    e_pa     = qmof_props.get("pbe_energy_per_atom")
    mag      = qmof_props.get("pbe_absolute_magnetization")

    if pbe_gap  is not None: print(f"  PBE   band gap : {pbe_gap:.3f} eV")
    if hle_gap  is not None: print(f"  HLE17 band gap : {hle_gap:.3f} eV  (HSE-like correction)")
    if e_pa     is not None: print(f"  PBE energy/atom: {e_pa:.4f} eV")
    if mag      is not None: print(f"  |magnetization|: {mag:.2f} μB")

    e_cp2k_pa = e_cp2k_bulk / len(atoms_cp2k)
    print(f"\n  CP2K energy/atom (this notebook, {CP2K_XC}): {e_cp2k_pa:.4f} eV")
    if e_pa is not None and CP2K_XC == "PBE":
        print(f"  Δ vs QMOF PBE  : {e_cp2k_pa - e_pa:+.4f} eV/atom "
              "(some offset is expected — different basis set/pseudopotential/relaxation)")

    print()
    print("  Literature optical band gap : ~2.3 eV (diffuse reflectance spectroscopy)")
    print("  Extracting a CP2K band gap needs an explicit &PDOS / &BAND_STRUCTURE "
          "post-processing run (k-point sampling) — not computed in this single-point cell.")
else:
    print("No pre-computed QMOF properties available for comparison.")
    print(f"  CP2K energy/atom : {e_cp2k_bulk/len(atoms_cp2k):.4f} eV  ({CP2K_XC})")
    print(f"  Literature (exp) : ~2.3 eV band gap (diffuse reflectance spectroscopy)")


## Section 7 · Pore and Structural Analysis

In [ ]:
# ── Pore geometry analysis ────────────────────────────────────────────────────
#
# For MOF-74-Mn, the characteristic pore diameter is ~11 Å.
# We can estimate it from the crystal structure by finding the largest sphere
# that fits in the void, or from the Mn–Mn inter-rod distance.

sym_arr = atoms_relax.get_chemical_symbols()
pos_arr = atoms_relax.get_positions()

# Mn–Mn distances (across the pore)
mn_idx  = [i for i, s in enumerate(sym_arr) if s == "Mn"]
mn_pos  = pos_arr[mn_idx]

print("── Structural Analysis ────────────────────────────────────")
print(f"  Total atoms  : {len(atoms_relax)}")

if len(mn_idx) >= 2:
    from itertools import combinations
    mn_dists = sorted(
        [(i, j, np.linalg.norm(mn_pos[i]-mn_pos[j]))
         for i, j in combinations(range(len(mn_idx)), 2)],
        key=lambda x: x[2]
    )
    print(f"  Mn count     : {len(mn_idx)}")
    print(f"  Shortest Mn–Mn: {mn_dists[0][2]:.3f} Å")
    print(f"  Mn–Mn distances (up to 8 Å):")
    for (i, j, d) in mn_dists:
        if d < 8.0:
            print(f"    Mn{mn_idx[i]}–Mn{mn_idx[j]}: {d:.3f} Å")
elif len(mn_idx) == 1:
    print(f"  Single Mn in cell (simplified model)")
    # Estimate pore from max distance between atoms
    max_span = np.ptp(pos_arr, axis=0)
    print(f"  Molecular span: {max_span}")

# Coordination number analysis for Mn
print()
for mi in mn_idx[:3]:   # first 3 Mn atoms
    mn_p = pos_arr[mi]
    coord_atoms = []
    for j, (s, p) in enumerate(zip(sym_arr, pos_arr)):
        if j != mi:
            d = np.linalg.norm(p - mn_p)
            if d < 2.6:   # Mn–O typical < 2.5 Å
                coord_atoms.append((s, d))
    coord_atoms.sort(key=lambda x: x[1])
    print(f"  Mn({mi}) coordination ({len(coord_atoms)} bonds):")
    for (s, d) in coord_atoms:
        print(f"    Mn–{s}: {d:.3f} Å")

# Volume per Mn (crude estimate of pore volume)
vol  = atoms_relax.get_volume()
n_mn = len(mn_idx) if mn_idx else 1
print(f"\n  Cell volume      : {vol:.2f} Å³")
print(f"  Volume/Mn        : {vol/n_mn:.2f} Å³/Mn  (crude proxy for pore volume)")


## Summary

In [ ]:
print("═"*60)
print("  SUMMARY: MOF-74-Mn Bulk Periodic Structure")
print("═"*60)
print(f"  Atoms in cell    : {len(atoms_mof)}")
print(f"  PBC              : {atoms_mof.get_pbc().tolist()}")
lengths = atoms_mof.cell.lengths(); angles = atoms_mof.cell.angles()
print(f"  a,b,c (Å)       : {lengths[0]:.2f}  {lengths[1]:.2f}  {lengths[2]:.2f}")
print(f"  Volume           : {atoms_mof.get_volume():.2f} Å³")
print()
print(f"  MACE energy      : {e_mace:.4f} eV")
print(f"  MACE/atom        : {e_mace/len(atoms_mof):.4f} eV/atom")
print(f"  MACE stress (hyd): {-np.trace(st_mace * 160.2)/3:.3f} GPa")
print(f"  CP2K energy      : {e_cp2k_bulk:.4f} eV  ({CP2K_XC}, periodic DFT)")
print(f"  CP2K/atom        : {e_cp2k_bulk/len(atoms_cp2k):.4f} eV/atom")
print(f"    (Note: band gap needs a separate CP2K PDOS/band-structure run)")
print()
import glob
for f in sorted(glob.glob(f"{WORK_DIR}/*")):
    print(f"  {f}")
print("═"*60)
